<a href="https://colab.research.google.com/github/Maike-Simoncini/Assistente-de-Voz-Multi-idiomas-integrado-com-OpenAI-Whisper-e-Google-Gemini/blob/main/Assistente_de_Voz_Multi_idiomas_integrado_com_OpenAI_Whisper_e_Google_Gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [60]:
language = 'pt'

# 1. Gravação de Áudio Com Python (e Uma Pitada de JavaScript) 🎤

In [61]:
# Referência: https://gist.github.com/korakot/c21c3476c024ad6d56d5f48b0bca92be

from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

# Código JavaScript para gravar áudio do usuário usando a "MediaStream Recording API"
RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=5):
  # Executa o código JavaScript para gravar o áudio
  display(Javascript(RECORD))
  # Recebe o áudio gravado como resultado do JavaScript
  js_result = output.eval_js('record(%s)' % (sec * 1000))
   # Decodifica o áudio em base64
  audio = b64decode(js_result.split(',')[1])
  # Salva o áudio em um arquivo
  file_name = 'request_audio.wav'
  with open(file_name, 'wb') as f:
    f.write(audio)
  # Retorna o caminho do arquivo de áudio (pasta padrão do Google Colab)
  return f'/content/{file_name}'

# Grava o áudio do usuário por um tempo determinado (padrão 5 segundos)
print('Ouvindo...\n')
record_file = record()

# Exibe o áudio gravado
display(Audio(record_file, autoplay=False))

Ouvindo...



<IPython.core.display.Javascript object>

# 2. Reconhecimento de Fala com Whisper (OpenAI) 🧠

In [62]:
!pip install git+https://github.com/openai/whisper.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [63]:
import whisper

# Selecione o modelo do Whisper que melhor atenda às suas necessidades:
# https://github.com/openai/whisper#available-models-and-languages
model = whisper.load_model("small")

# Transcreve o audio gravado anteriormente.
result = model.transcribe(record_file, fp16=False, language=language)
transcription = result["text"]
print(transcription)

 Quanto que é 2 mais 2?


In [64]:
import os
from google.colab import userdata

# Documentação Oficial da API Gemini: https://ai.google.dev/docs

# Para usar a API Gemini, você precisará de uma chave de API.
# 1. Crie uma conta no Google AI Studio (https://aistudio.google.com/)
# 2. Acesse a seção 'Get API Key'
# 3. Clique em 'Create API Key'

# No Colab, adicione a chave ao gerenciador de segredos
# (o ícone de chave no painel esquerdo) e chame-a de `GOOGLE_API_KEY`.
# Isso garante que sua chave não seja exposta diretamente no código.

# Obtém a chave de API dos segredos do Colab
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY

# 3. Integração com a API do Gemini 💬

In [65]:
import google.generativeai as genai
import os

# Configura a chave de API do Gemini usando a variável de ambiente 'GOOGLE_API_KEY'
genai.configure(api_key=os.environ.get('GOOGLE_API_KEY'))

# Opcional: Lista os modelos disponíveis para o método 'generateContent'
# Descomente as linhas abaixo se precisar verificar os modelos novamente.
# print('Listando modelos disponíveis para generateContent:')
# for m in genai.list_models():
#   if 'generateContent' in m.supported_generation_methods:
#     print(m.name)

# Inicializa o modelo Gemini com o modelo 'gemini-2.5-flash'
gemini_model = genai.GenerativeModel('gemini-2.5-flash')

# Envia a transcrição do áudio para o modelo Gemini
response = gemini_model.generate_content(transcription)

# Obtém a resposta gerada pelo Gemini
chatgpt_response = response.text
print(chatgpt_response)

2 mais 2 é **4**.


# 4. Sintetizando a Resposta do ChatGPT Como Voz (gTTS) 🔊

In [66]:
!pip install gTTS

In [67]:
from gtts import gTTS
from IPython.display import Audio, display

# Remove os asteriscos da resposta para que o gTTS não os leia.
cleaned_chatgpt_response = chatgpt_response.replace('**', '')

# Cria um objeto gTTS com a resposta gerada pelo Gemini e a língua que será sintetizada em voz (variável "language").
gtts_object = gTTS(text=cleaned_chatgpt_response, lang=language, slow=False)

# Salva o áudio da resposta no arquivo especificado (pasta padrão do Google Colab)
response_audio = "/content/response_audio.wav"
gtts_object.save(response_audio)

# Reproduz o áudio da resposta salvo no arquivo
display(Audio(response_audio, autoplay=True))